In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM
processed_dpath = '/Users/oxide/Documents/research/orenstein/code/P288FinalProject/processed_data/'

In [14]:
##
## LSTM goal: predict erruption activity over the next predict_window minutes based
##            on erruption activity and other measurements over the last train_window minutes
##
predict_window = 1*24*60 # 1 day
train_window = 7*24*60 # 1 week

# convert series to supervised learning
def series_to_supervised(data, n_in=predict_window, n_out=train_window, dropnan=True):
	"""
	data: Sequence of observations as a list or 2D NumPy array. Required.
	n_in: Number of lag observations as input (X). Values may be between [1..len(data)] Optional. Defaults to 1.
	n_out: Number of observations as output (y). Values may be between [0..len(data)-1]. Optional. Defaults to 1.
	"""
	n_vars = 1 if type(data) is list else data.shape[1]
	df = pd.DataFrame(data)
	cols, names = list(), list()
	
	# input sequence (t-n, ... t-1)
	for i in range(n_in, 0, -1):
		cols.append(df.shift(i))
		names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]

	# forecast sequence (t, t+1, ... t+n)
	for i in range(0, n_out):
		cols.append(df.shift(-i))
		if i == 0:
			names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
		else:
			names += [('var%d(t+%d)' % (j+1, i)) for j in range(n_vars)]
	# put it all together
	agg = pd.concat(cols, axis=1)
	agg.columns = names
	# drop rows with NaN values
	if dropnan:
		agg.dropna(inplace=True)
	return agg

In [15]:
###
### Preprocessing
###

# load data, keep only certain variables, and fill Nans
vars = ['Erruption Activity', 'VPCC RSAM', 'VPPC RSAM', 'VPNC RSAM', 'VPRS RSAM', 'VPRS Field E', 'VPRS Field N', 'VPRS Field Z', 'CO2 Concentration']
data = pd.read_csv(processed_dpath+f'dataInterpolated.csv', index_col=0)
data = data[vars]
data.fillna(0, inplace=True)
data.name = 'lstmData'
data.to_csv(processed_dpath+f'{data.name}.csv')

# normalize features
values = data.values
scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(values)

# reframe as supervised learning
reframed = series_to_supervised(scaled, 1, 1)

In [16]:
reframed

,var1(t-1),var2(t-1),var3(t-1),var4(t-1),var5(t-1),var6(t-1),var7(t-1),var8(t-1),var9(t-1),var1(t),var2(t),var3(t),var4(t),var5(t),var6(t),var7(t),var8(t),var9(t)
1,0.004153,0.003712,0.015309,0.008843,0.008848,0.457519,0.742275,0.000185,0.384938,0.004153,0.003618,0.015622,0.008825,0.008762,0.456156,0.742275,0.000185,0.387442
2,0.004153,0.003618,0.015622,0.008825,0.008762,0.456156,0.742275,0.000185,0.387442,0.004153,0.003525,0.015935,0.008807,0.008677,0.455248,0.742626,0.000185,0.389947
3,0.004153,0.003525,0.015935,0.008807,0.008677,0.455248,0.742626,0.000185,0.389947,0.004153,0.003432,0.016248,0.008789,0.008591,0.453657,0.743329,0.000185,0.393135
4,0.004153,0.003432,0.016248,0.008789,0.008591,0.453657,0.743329,0.000185,0.393135,0.004153,0.003338,0.016560,0.008771,0.008506,0.450477,0.742626,0.000185,0.393819
5,0.004153,0.003338,0.016560,0.008771,0.008506,0.450477,0.742626,0.000185,0.393819,0.004153,0.003245,0.016873,0.008754,0.008420,0.446842,0.741924,0.000246,0.394503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76316,0.004153,0.007373,0.168550,0.069996,0.004530,0.337119,0.586728,0.999508,0.997770,0.004153,0.006873,0.169363,0.069764,0.004451,0.336665,0.584972,0.999631,0.998216
76317,0.004153,0.006873,0.169363,0.069764,0.004451,0.336665,0.584972,0.999631,0.998216,0.004153,0.006372,0.170176,0.069531,0.004371,0.335075,0.583919,0.999692,0.998662
76318,0.004153,0.006372,0.170176,0.069531,0.004371,0.335075,0.583919,0.999692,0.998662,0.004153,0.005871,0.170990,0.069298,0.004291,0.334393,0.582514,0.999815,0.999108
76319,0.004153,0.005871,0.170990,0.069298,0.004291,0.334393,0.582514,0.999815,0.999108,0.004153,0.005370,0.171803,0.069065,0.004211,0.334166,0.581812,0.999938,0.999554
